# Hospitality PMS LLM — Evaluation Harness (Colab)

Runs the full evaluation matrix on a free T4 GPU.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Have `vectorstore.zip` ready to upload (77MB) — prepared locally with:
   ```bash
   cd hospitality-pms-llm/output && zip -r vectorstore.zip vectorstore/
   ```

In [2]:
# 1. Clone repo from GitHub
!git clone https://github.com/pradray/hospitality-pms-llm.git
%cd hospitality-pms-llm

Cloning into 'hospitality-pms-llm'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 59 (delta 19), reused 53 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 3.21 MiB | 15.42 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/hospitality-pms-llm


In [ ]:
# 2. Install dependencies
!pip install -q llama-cpp-python \
    chromadb==1.5.9 \
    sentence-transformers==5.6.0 \
    huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 MB 10.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# 3. Download models from HuggingFace
!huggingface-cli download Qwen/Qwen2.5-3B-Instruct-GGUF qwen2.5-3b-instruct-q4_k_m.gguf --local-dir ./models
!huggingface-cli download Qwen/Qwen2.5-7B-Instruct-GGUF \
    qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf \
    qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf \
    --local-dir ./models

In [ ]:
# 4. Upload vectorstore (only thing not in git)
from google.colab import files
import os, shutil

print("Upload vectorstore.zip (77MB)")
uploaded = files.upload()

os.makedirs('output', exist_ok=True)
shutil.move('vectorstore.zip', 'output/vectorstore.zip')
!cd output && unzip -qo vectorstore.zip && rm vectorstore.zip
print("Vectorstore extracted.")

In [ ]:
# 5. Verify setup
import os, json

print("Models:")
for f in os.listdir('models'):
    if f.endswith('.gguf'):
        size_gb = os.path.getsize(f'models/{f}') / 1e9
        print(f"  {f} ({size_gb:.1f} GB)")

print(f"\nVectorstore exists: {os.path.exists('output/vectorstore')}")
print(f"Benchmark dev exists: {os.path.exists('data/benchmark_dev.jsonl')}")

with open('data/benchmark_dev.jsonl') as f:
    tasks = [json.loads(l) for l in f]
print(f"Benchmark tasks: {len(tasks)}")

In [ ]:
# 6. Import pipeline and run eval
import sys
sys.path.insert(0, 'src')

import json, time, os
from datetime import datetime
from rag_pipeline import RAGPipeline, EVAL_CONFIGS


def run_eval(config_name, tasks):
    print(f'\n{"="*60}')
    print(f'Config: {config_name}')
    print(f'{"="*60}')

    pipeline = RAGPipeline.from_config(config_name)
    results = []

    for i, task in enumerate(tasks, 1):
        q = task['question']
        print(f'  [{i}/{len(tasks)}] {task["id"]}: {q[:70]}...')

        start = time.time()
        response = pipeline.query(q)
        elapsed = time.time() - start

        results.append({
            'task_id': task['id'],
            'config': config_name,
            'model': response.model,
            'use_rag': pipeline.use_rag,
            'question': q,
            'expected_answer': task.get('expected_answer', ''),
            'generated_answer': response.answer,
            'category': task.get('category', ''),
            'module': task.get('module', ''),
            'difficulty': task.get('difficulty', ''),
            'num_chunks_retrieved': len(response.chunks),
            'chunk_distances': [c.distance for c in response.chunks],
            'latency_seconds': round(elapsed, 2),
        })
        print(f'         {elapsed:.1f}s | {len(response.answer)} chars')

    del pipeline
    import gc; gc.collect()
    return results


# Load benchmark
with open('data/benchmark_dev.jsonl') as f:
    tasks = [json.loads(l) for l in f]
print(f'Loaded {len(tasks)} tasks')

In [ ]:
# 7. Run all 4 local configs

configs_to_run = ['3B-base', '3B-RAG', '7B-base', '7B-RAG']
all_results = []

for config_name in configs_to_run:
    try:
        results = run_eval(config_name, tasks)
        all_results.extend(results)

        # Save after each config (in case of crash)
        os.makedirs('output/eval_results', exist_ok=True)
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        partial_path = f'output/eval_results/eval_dev_{config_name}_{ts}.jsonl'
        with open(partial_path, 'w') as f:
            for r in results:
                f.write(json.dumps(r) + '\n')
        print(f'  Saved {len(results)} results → {partial_path}')

    except Exception as e:
        print(f'ERROR running {config_name}: {e}')
        import traceback; traceback.print_exc()

# Save combined results
combined_path = f'output/eval_results/eval_dev_all_{ts}.jsonl'
with open(combined_path, 'w') as f:
    for r in all_results:
        f.write(json.dumps(r) + '\n')

print(f'\nAll results: {len(all_results)} total → {combined_path}')

# Summary
from collections import defaultdict
by_config = defaultdict(list)
for r in all_results:
    by_config[r['config']].append(r['latency_seconds'])

print(f'\n{"Config":<15} {"Tasks":>6} {"Avg Latency":>12}')
print('-' * 35)
for cfg in configs_to_run:
    lats = by_config[cfg]
    if lats:
        print(f'{cfg:<15} {len(lats):>6} {sum(lats)/len(lats):>10.1f}s')

In [ ]:
# 8. Download results
from google.colab import files

# Download the combined file
files.download(combined_path)

# Also download per-config files as backup
import glob
for f in glob.glob('output/eval_results/eval_dev_*.jsonl'):
    print(f'Available: {f}')